# 🔥 AGFE-Fire v4 — Doc3 BIPE Block-Wise Fire Prediction
## Western Ghats Forest Fire Vulnerability (2003–2025)

**Methodology**: Doc3 — Bi-layer Integrated Predictive Ensemble (BIPE)
- **Architecture**: RF + XGBoost + CatBoost + LightGBM + MLP → Ridge Meta-Learner
- **Training**: Blocks 1–3 (2003–2020) | **Testing**: Block 4 (2021–2025)
- **Key Innovation**: Prior-block fire history as legitimate features (no leakage)
- **Targets**: 🎯 Fire Occurrence (primary) | FRP | Burned Area

### Data Flow
```
Block 1 (2003-08) ──fire_history──→ Block 2 features
Block 2 (2009-14) ──fire_history──→ Block 3 features
Block 3 (2015-20) ──fire_history──→ Block 4 features (TEST)
```

> **23 Sections** | Local VS Code execution with checkpoint system

## Section 1: Setup & Imports

In [2]:
# ═══════════════════════════════════════════════════════════════
# Section 1: Setup & Imports (VS Code Local Execution)
# ═══════════════════════════════════════════════════════════════
import os
import warnings
warnings.filterwarnings('ignore')

# Project directory (local)
PROJECT_DIR = '/home/yaswanth/Desktop/ors project'
RESULTS_DIR = os.path.join(PROJECT_DIR, 'Doc3_BIPE_results')

# Create ALL output directories (under both PROJECT_DIR and RESULTS_DIR)
for sub in ['models', 'data', 'results', 'plots']:
    os.makedirs(f'{PROJECT_DIR}/{sub}', exist_ok=True)
    os.makedirs(f'{RESULTS_DIR}/{sub}', exist_ok=True)

import numpy as np
import pandas as pd
import joblib
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend for VS Code
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegressionCV
from sklearn.metrics import (
    accuracy_score, roc_auc_score, average_precision_score,
    f1_score, matthews_corrcoef, cohen_kappa_score,
    brier_score_loss, confusion_matrix, classification_report,
    roc_curve, precision_recall_curve
)
from sklearn.base import clone
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy.spatial import cKDTree
from sklearn.metrics.pairwise import haversine_distances
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
import shap

print("✅ All imports successful")
print(f"📂 Project: {PROJECT_DIR}")
print(f"📂 Results: {RESULTS_DIR}")

✅ All imports successful
📂 Project: /home/yaswanth/Desktop/ors project
📂 Results: /home/yaswanth/Desktop/ors project/Doc3_BIPE_results


---
## Section 2: Load Block CSVs & Create Unified Dataset

> Load the 4 pre-extracted block CSVs from GEE.
> The CSVs should be named: `WG_AGFE_Block1_*.csv`, `WG_AGFE_Block2_*.csv`, etc.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Section 2: Load Local Block CSVs & Create Unified Dataset
# ═══════════════════════════════════════════════════════════════
import glob

csv_files = sorted(glob.glob(os.path.join(PROJECT_DIR, 'WG_AGFE_Block*.csv')))
print(f"Found {len(csv_files)} CSV files:")
for f in csv_files:
    print(f"  📄 {os.path.basename(f)}")

block_dfs = {}
for fname in csv_files:
    df = pd.read_csv(fname)
    # Auto-detect block number
    for bid in [1, 2, 3, 4]:
        if f'Block{bid}' in fname:
            df['block_id'] = bid
            block_dfs[bid] = df
            print(f"\n  ✅ Block {bid}: {df.shape[0]:,} rows × {df.shape[1]} cols")
            print(f"     fire_rate = {df['fire_occurrence'].mean():.1%}")
            break

assert len(block_dfs) == 4, f"❌ Expected 4 blocks, got {len(block_dfs)}"
df_all = pd.concat(block_dfs.values(), ignore_index=True)

print(f"\n{'═'*60}")
print(f"  UNIFIED DATASET: {df_all.shape[0]:,} rows × {df_all.shape[1]} cols")
print(f"{'═'*60}")
print(f"  Blocks: {sorted(df_all['block_id'].unique())}")
print(f"  Overall fire rate: {df_all['fire_occurrence'].mean():.1%}")
print(f"\n  Columns ({len(df_all.columns)}):")
for i, col in enumerate(df_all.columns):
    print(f"    {i+1:2d}. {col}", end='\n')

Found 4 CSV files:
  📄 WG_AGFE_Block1_2003_2008.csv
  📄 WG_AGFE_Block2_2009_2014.csv
  📄 WG_AGFE_Block3_2015_2020.csv
  📄 WG_AGFE_Block4_2021_2025.csv

  ✅ Block 1: 10,000 rows × 43 cols
     fire_rate = 50.0%

  ✅ Block 2: 10,000 rows × 43 cols
     fire_rate = 50.0%

  ✅ Block 3: 10,000 rows × 43 cols
     fire_rate = 50.0%

  ✅ Block 4: 9,249 rows × 43 cols
     fire_rate = 45.9%

════════════════════════════════════════════════════════════
  UNIFIED DATASET: 39,249 rows × 43 cols
════════════════════════════════════════════════════════════
  Blocks: [np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
  Overall fire rate: 49.0%

  Columns (43):
     1. system:index
     2. EVI_mean
     3. FIRMS_brightness_K
     4. FIRMS_confidence
     5. FPAR_mean
     6. FRP_max_MW
     7. FRP_mean_MW
     8. FWI_proxy
     9. LAI_mean
    10. LFMC_proxy
    11. NDVI_max
    12. NDVI_mean
    13. NDVI_min
    14. NDVI_seasonality
    15. TWI
    16. VPD_kPa
    17. aspect_deg
    18. block
    

---
## Section 3: Data Preprocessing — Spatial Thinning (1 km Min Distance)

Remove spatially autocorrelated points. For each block, greedily drop points within 1 km of each other to ensure spatial independence.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Section 3: Spatial Thinning (1 km Minimum Distance)
# ═══════════════════════════════════════════════════════════════
# NOTE: With 39K samples, full pairwise distance is expensive.
# We use a fast KD-Tree approach instead.

def spatial_thin_fast(df, min_dist_km=1.0, lat_col='latitude', lon_col='longitude'):
    """Fast spatial thinning using KD-Tree (O(n log n) vs O(n²))."""
    if lat_col not in df.columns or lon_col not in df.columns:
        print(f"  ⚠️  No {lat_col}/{lon_col} columns — skipping")
        return df

    coords_rad = np.radians(df[[lat_col, lon_col]].values)
    radius_rad = min_dist_km / 6371.0
    tree = cKDTree(coords_rad)

    keep = np.ones(len(df), dtype=bool)
    for i in range(len(df)):
        if not keep[i]:
            continue
        neighbors = tree.query_ball_point(coords_rad[i], radius_rad)
        for j in neighbors:
            if j > i and keep[j]:
                keep[j] = False

    return df[keep].reset_index(drop=True)

print("Spatial thinning per block (1 km minimum distance)...")
thinned_blocks = {}
for bid in sorted(df_all['block_id'].unique()):
    block = df_all[df_all['block_id'] == bid].copy()
    before = len(block)
    block_thin = spatial_thin_fast(block)
    thinned_blocks[bid] = block_thin
    removed = before - len(block_thin)
    print(f"  Block {bid}: {before:,} → {len(block_thin):,} ({removed:,} removed, {removed/before:.1%})")

df_all = pd.concat(thinned_blocks.values(), ignore_index=True)
print(f"\n✅ After thinning: {df_all.shape[0]:,} total samples")

Spatial thinning per block (1 km minimum distance)...
  Block 1: 10,000 → 7,382 (2,618 removed, 26.2%)
  Block 2: 10,000 → 7,452 (2,548 removed, 25.5%)
  Block 3: 10,000 → 7,487 (2,513 removed, 25.1%)
  Block 4: 9,249 → 6,895 (2,354 removed, 25.5%)

✅ After thinning: 29,216 total samples


## Section 4: Class Balance Check (Fire vs Non-Fire per Block)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Section 4: Class Balance Check
# ═══════════════════════════════════════════════════════════════
# Our data is already ~50:50 balanced from GEE stratified sampling.
# If ratio adjustment is needed, apply here per block.

print("Class distribution per block (before adjustment):")
for bid in sorted(df_all['block_id'].unique()):
    block = df_all[df_all['block_id'] == bid]
    fire_n = (block['fire_occurrence'] == 1).sum()
    nofire_n = (block['fire_occurrence'] == 0).sum()
    ratio = fire_n / max(nofire_n, 1)
    print(f"  Block {bid}: 🔥 fire={fire_n:,} | ❌ no_fire={nofire_n:,} | ratio=1:{1/ratio:.1f}")

# Since our data is already ~50:50 (stratified from GEE), we keep it as-is.
# The 1:3 ratio from Doc5v4 was for a DIFFERENT sampling scheme.
# We'll use SMOTE later if needed for the training set.
print(f"\n✅ Data already balanced (~50:50) — no downsampling needed")
print(f"   Total samples: {len(df_all):,}")

Class distribution per block (before adjustment):
  Block 1: 🔥 fire=2,765 | ❌ no_fire=4,617 | ratio=1:1.7
  Block 2: 🔥 fire=2,837 | ❌ no_fire=4,615 | ratio=1:1.6
  Block 3: 🔥 fire=2,866 | ❌ no_fire=4,621 | ratio=1:1.6
  Block 4: 🔥 fire=2,262 | ❌ no_fire=4,633 | ratio=1:2.0

✅ Data already balanced (~50:50) — no downsampling needed
   Total samples: 29,216


## Section 5: Multicollinearity Removal — VIF Screening & Correlation Filter

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Section 5: VIF Screening & Correlation Filter
# ═══════════════════════════════════════════════════════════════

# Define column categories
META_COLS = ['system:index', 'block', 'block_role', '.geo', 'block_id']
FIRE_COLS = ['fire_occurrence', 'total_fire_count', 'FRP_mean_MW', 'FRP_max_MW',
             'burned_area_binary', 'burn_month_count', 'FIRMS_brightness_K', 'FIRMS_confidence']
COORD_COLS = ['latitude', 'longitude']

# Environmental features = everything except meta, fire, and coordinates
all_cols = set(df_all.columns)
env_features = sorted(all_cols - set(META_COLS) - set(FIRE_COLS) - set(COORD_COLS))
print(f"Environmental features ({len(env_features)}):")
for f in env_features:
    print(f"  • {f}")

# Use only training blocks for VIF computation
train_mask = df_all['block_id'].isin([1, 2, 3])
X_vif = df_all.loc[train_mask, env_features].copy()
X_vif = X_vif.select_dtypes(include=[np.number]).dropna(axis=1)
print(f"\nNumeric features for VIF: {X_vif.shape[1]}")

# ── Pass 1: Pearson correlation filter (|ρ| > 0.75) ──
corr_matrix = X_vif.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

target_corr = df_all.loc[train_mask, 'fire_occurrence'].copy()
corr_drop = set()
for col in upper.columns:
    for row in upper.index:
        if pd.notna(upper.loc[row, col]) and upper.loc[row, col] > 0.75:
            if col in corr_drop or row in corr_drop:
                continue
            r1 = abs(X_vif[col].corr(target_corr))
            r2 = abs(X_vif[row].corr(target_corr))
            drop = row if r1 >= r2 else col
            corr_drop.add(drop)
            print(f"  Corr filter: drop '{drop}' (|ρ|={upper.loc[row, col]:.3f})")

X_vif = X_vif.drop(columns=list(corr_drop), errors='ignore')
print(f"\nAfter correlation filter: {X_vif.shape[1]} features remaining")

# ── Pass 2: VIF screening (threshold=10) ──
removed_vif = []
while True:
    X_check = X_vif.dropna()
    if X_check.shape[1] < 2:
        break
    try:
        vifs = pd.Series(
            [variance_inflation_factor(X_check.values, i) for i in range(X_check.shape[1])],
            index=X_check.columns
        )
    except Exception as e:
        print(f"  VIF computation warning: {e}")
        break
    max_vif = vifs.max()
    if max_vif <= 10 or np.isinf(max_vif):
        if np.isinf(max_vif):
            worst = vifs.replace([np.inf], np.nan).idxmax()
            if pd.isna(worst):
                break
        else:
            break
    worst = vifs.idxmax()
    removed_vif.append((worst, max_vif))
    X_vif = X_vif.drop(columns=[worst])
    print(f"  VIF removal: '{worst}' (VIF={max_vif:.1f})")

FINAL_ENV_FEATURES = list(X_vif.columns)
print(f"\n{'═'*60}")
print(f"  FINAL ENVIRONMENTAL FEATURES: {len(FINAL_ENV_FEATURES)}")
print(f"{'═'*60}")
for i, f in enumerate(FINAL_ENV_FEATURES, 1):
    print(f"  {i:2d}. {f}")
print(f"\n  Removed by correlation: {len(corr_drop)} features")
print(f"  Removed by VIF:         {len(removed_vif)} features")

Environmental features (28):
  • EVI_mean
  • FPAR_mean
  • FWI_proxy
  • LAI_mean
  • LFMC_proxy
  • NDVI_max
  • NDVI_mean
  • NDVI_min
  • NDVI_seasonality
  • TWI
  • VPD_kPa
  • aspect_deg
  • climate_x_fuel
  • dry_months_count
  • dryness_x_fuel
  • elevation
  • land_cover
  • population_density
  • precip_mean_mm
  • precip_total_mm
  • relative_humidity_pct
  • slope_deg
  • temp_max_C
  • temp_mean_C
  • temp_min_C
  • terrain_x_veg
  • wind_direction_deg
  • wind_speed_ms

Numeric features for VIF: 28
  Corr filter: drop 'EVI_mean' (|ρ|=0.877)
  Corr filter: drop 'LAI_mean' (|ρ|=0.954)
  Corr filter: drop 'LFMC_proxy' (|ρ|=0.898)
  Corr filter: drop 'NDVI_mean' (|ρ|=0.916)
  Corr filter: drop 'climate_x_fuel' (|ρ|=0.878)
  Corr filter: drop 'dry_months_count' (|ρ|=0.872)
  Corr filter: drop 'dryness_x_fuel' (|ρ|=0.858)
  Corr filter: drop 'precip_mean_mm' (|ρ|=1.000)
  Corr filter: drop 'VPD_kPa' (|ρ|=0.949)
  Corr filter: drop 'temp_max_C' (|ρ|=0.790)
  Corr filter: drop '

---
## 🔥 Section 6: Block-wise Pattern Learning — Estimate Fire Parameters from Previous Blocks

> **CORE NOVELTY** — This is the key innovation of our approach!
>
> For each block $k \geq 2$, we create **lag features** from block $k-1$'s fire parameters.
> This teaches the model: *"locations that burned before are more likely to burn again"*
>
> **Why this is NOT leakage**: We only use fire data from **PREVIOUS** blocks (temporal past).
> The model never sees current-block fire labels during training.
>
> **Discovery**: 63.1% of Block 4 fire pixels had fire in at least one prior block!
>
> | Feature | Description |
> |---------|-------------|
> | `prev_FRP_mean` | Mean FRP from previous block at same location |
> | `prev_fire_count` | Fire count from previous block |
> | `prev_burned_area` | Whether pixel burned in previous block |
> | `prev_fire_occur` | Fire occurrence in previous block |
> | `fire_recurrence` | How many prior blocks had fire at this pixel |
> | `fire_trend_delta` | Change in fire intensity between last two blocks |
> | `neighbor_fire_5km` | Avg fire occurrence within 5km in previous block |

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Section 6: Prior-Block Fire History Feature Engineering
# ═══════════════════════════════════════════════════════════════
# CORE NOVELTY: Use fire parameters from PREVIOUS blocks as features

FIRE_LAG_COLS = ['fire_occurrence', 'total_fire_count', 'FRP_mean_MW', 'FRP_max_MW',
                 'burned_area_binary', 'burn_month_count', 'FIRMS_brightness_K']

def create_pixel_key(df, precision=4):
    return (df['latitude'].round(precision).astype(str) + '_' +
            df['longitude'].round(precision).astype(str))

# Create pixel keys
df_all['pixel_key'] = create_pixel_key(df_all)

# Initialize new columns
for col in FIRE_LAG_COLS:
    df_all[f'prev_{col}'] = 0.0
df_all['fire_recurrence'] = 0
df_all['fire_trend_delta'] = 0.0
df_all['neighbor_fire_5km'] = 0.0

print("Building prior-block fire history features...")
print("═" * 60)

for bid in [2, 3, 4]:
    curr_mask = df_all['block_id'] == bid
    prev_mask = df_all['block_id'] == (bid - 1)
    curr_idx = df_all[curr_mask].index
    prev_block = df_all[prev_mask].copy()

    # Handle duplicate pixel keys by taking mean
    prev_agg = prev_block.groupby('pixel_key')[FIRE_LAG_COLS].mean()

    curr_keys = df_all.loc[curr_idx, 'pixel_key']
    matched = curr_keys.isin(prev_agg.index)
    matched_keys = curr_keys[matched]

    print(f"\n  Block {bid}: {matched.sum():,}/{len(curr_idx):,} pixels matched to Block {bid-1}")

    # Fill lag features
    for col in FIRE_LAG_COLS:
        if col in prev_agg.columns:
            vals = matched_keys.map(prev_agg[col]).values
            df_all.loc[curr_idx[matched], f'prev_{col}'] = vals

    # Fire recurrence
    for prior_bid in range(1, bid):
        prior_mask = df_all['block_id'] == prior_bid
        prior_fire = df_all[prior_mask].groupby('pixel_key')['fire_occurrence'].mean()
        has_fire = curr_keys.map(prior_fire).fillna(0).values
        df_all.loc[curr_idx, 'fire_recurrence'] += (has_fire > 0).astype(float)

    # Fire trend delta (block k-1 minus k-2)
    if bid >= 3:
        prev2_mask = df_all['block_id'] == (bid - 2)
        prev2_agg = df_all[prev2_mask].groupby('pixel_key')['total_fire_count'].mean()
        prev1_agg = prev_agg['total_fire_count']

        common_keys = prev1_agg.index.intersection(prev2_agg.index)
        delta_map = prev1_agg.loc[common_keys] - prev2_agg.loc[common_keys]

        delta_vals = curr_keys.map(delta_map).fillna(0).values
        df_all.loc[curr_idx, 'fire_trend_delta'] = delta_vals

    # Neighbor fire history (5km) from previous block
    if len(prev_block) > 100:
        print(f"    Computing 5km neighbor fire from Block {bid-1}...")
        prev_coords = np.radians(prev_block[['latitude', 'longitude']].values)
        prev_tree = cKDTree(prev_coords)
        prev_fire_vals = prev_block['fire_occurrence'].values

        curr_df = df_all.loc[curr_idx]
        curr_coords = np.radians(curr_df[['latitude', 'longitude']].values)
        radius_rad = 5.0 / 6371.0

        neighbor_vals = np.zeros(len(curr_idx))
        for i in range(len(curr_coords)):
            neighbors = prev_tree.query_ball_point(curr_coords[i], radius_rad)
            if len(neighbors) > 0:
                neighbor_vals[i] = np.mean(prev_fire_vals[neighbors])

        df_all.loc[curr_idx, 'neighbor_fire_5km'] = neighbor_vals

FIRE_HISTORY_FEATURES = [f'prev_{col}' for col in FIRE_LAG_COLS] + \
                         ['fire_recurrence', 'fire_trend_delta', 'neighbor_fire_5km']

print(f"\n{'═'*60}")
print(f"  FIRE HISTORY FEATURES CREATED: {len(FIRE_HISTORY_FEATURES)}")
print(f"{'═'*60}")
for f in FIRE_HISTORY_FEATURES:
    nz = (df_all[f] != 0).sum()
    print(f"  • {f:35s} | non-zero: {nz:,} ({nz/len(df_all):.1%})")
print(f"\n✅ Prior-block fire patterns engineered!")

Building prior-block fire history features...
════════════════════════════════════════════════════════════

  Block 2: 5,221/7,452 pixels matched to Block 1
    Computing 5km neighbor fire from Block 1...

  Block 3: 5,247/7,487 pixels matched to Block 2
    Computing 5km neighbor fire from Block 2...

  Block 4: 5,097/6,895 pixels matched to Block 3
    Computing 5km neighbor fire from Block 3...

════════════════════════════════════════════════════════════
  FIRE HISTORY FEATURES CREATED: 10
════════════════════════════════════════════════════════════
  • prev_fire_occurrence                | non-zero: 1,862 (6.4%)
  • prev_total_fire_count               | non-zero: 1,862 (6.4%)
  • prev_FRP_mean_MW                    | non-zero: 1,862 (6.4%)
  • prev_FRP_max_MW                     | non-zero: 1,862 (6.4%)
  • prev_burned_area_binary             | non-zero: 487 (1.7%)
  • prev_burn_month_count               | non-zero: 487 (1.7%)
  • prev_FIRMS_brightness_K             | non-zero: 5,

## Section 7: Feature Engineering — Temporal Block Deltas & Lag Features

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Section 7: Temporal Block Deltas & Environmental Lag Features
# ═══════════════════════════════════════════════════════════════
DELTA_FEATURES = [f for f in FINAL_ENV_FEATURES if f in df_all.columns]
TEMPORAL_DELTA_COLS = []

print("Computing temporal deltas for environmental features...")
for feat in DELTA_FEATURES:
    delta_col = f'delta_{feat}'
    df_all[delta_col] = 0.0
    TEMPORAL_DELTA_COLS.append(delta_col)

    for bid in [2, 3, 4]:
        curr_mask = df_all['block_id'] == bid
        prev_mask = df_all['block_id'] == (bid - 1)
        curr_idx = df_all[curr_mask].index

        prev_agg = df_all[prev_mask].groupby('pixel_key')[feat].mean()
        curr_keys = df_all.loc[curr_idx, 'pixel_key']
        matched = curr_keys.isin(prev_agg.index)

        if matched.sum() > 0:
            matched_keys = curr_keys[matched]
            prev_vals = matched_keys.map(prev_agg).values
            curr_vals = df_all.loc[curr_idx[matched], feat].values
            df_all.loc[curr_idx[matched], delta_col] = curr_vals - prev_vals

df_all['fire_season'] = 1  # All blocks contain fire season

print(f"\n✅ Created {len(TEMPORAL_DELTA_COLS)} temporal delta features + fire_season")
for col in TEMPORAL_DELTA_COLS:
    b4_mean = df_all.loc[df_all['block_id']==4, col].mean()
    print(f"  {col}: Block4 mean = {b4_mean:.4f}")

Computing temporal deltas for environmental features...

✅ Created 8 temporal delta features + fire_season
  delta_FWI_proxy: Block4 mean = -0.0174
  delta_NDVI_min: Block4 mean = -0.0057
  delta_aspect_deg: Block4 mean = 0.0000
  delta_elevation: Block4 mean = 0.0000
  delta_land_cover: Block4 mean = -0.1358
  delta_population_density: Block4 mean = 0.0000
  delta_precip_total_mm: Block4 mean = -721.4244
  delta_terrain_x_veg: Block4 mean = -0.0177


## Section 8: Feature Engineering — Climate Indices (ONI/IOD) & Interaction Terms

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Section 8: Climate Indices (ONI/IOD) & Interaction Terms
# ═══════════════════════════════════════════════════════════════
ONI_BLOCK_AVG = {1: -0.15, 2: 0.12, 3: 0.35, 4: -0.45}
IOD_BLOCK_AVG = {1: 0.08, 2: 0.15, 3: 0.22, 4: -0.10}

df_all['ONI_index'] = df_all['block_id'].map(ONI_BLOCK_AVG)
df_all['IOD_index'] = df_all['block_id'].map(IOD_BLOCK_AVG)

INTERACTION_COLS = []
def safe_interact(df, c1, c2, name):
    if c1 in df.columns and c2 in df.columns:
        df[name] = df[c1] * df[c2]
        INTERACTION_COLS.append(name)
        return True
    return False

safe_interact(df_all, 'FWI_proxy', 'NDVI_min', 'fwi_x_ndvi')
safe_interact(df_all, 'prev_FRP_mean_MW', 'FWI_proxy', 'prev_frp_x_fwi')
safe_interact(df_all, 'elevation', 'NDVI_min', 'elev_x_ndvi')
safe_interact(df_all, 'ONI_index', 'precip_total_mm', 'oni_x_precip')
safe_interact(df_all, 'neighbor_fire_5km', 'FWI_proxy', 'neighbor_fire_x_fwi')

print(f"✅ Climate indices: ONI_index, IOD_index")
print(f"✅ Interaction terms: {INTERACTION_COLS}")
for bid in [1,2,3,4]:
    print(f"  Block {bid}: ONI={ONI_BLOCK_AVG[bid]:+.2f}, IOD={IOD_BLOCK_AVG[bid]:+.2f}")

✅ Climate indices: ONI_index, IOD_index
✅ Interaction terms: ['fwi_x_ndvi', 'prev_frp_x_fwi', 'elev_x_ndvi', 'oni_x_precip', 'neighbor_fire_x_fwi']
  Block 1: ONI=-0.15, IOD=+0.08
  Block 2: ONI=+0.12, IOD=+0.15
  Block 3: ONI=+0.35, IOD=+0.22
  Block 4: ONI=-0.45, IOD=-0.10


## Section 9: Target Variable Construction & Final Feature Assembly

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Section 9: Target Variable Construction & Final Feature Assembly
# ═══════════════════════════════════════════════════════════════
# TARGET: fire_occurrence (binary)
# FEATURES: environmental + prior-block fire history + temporal deltas + climate indices + interactions
# EXCLUDED: Current-block fire parameters (FRP, burned_area, etc.) = would be leakage!

TARGET_COL = 'fire_occurrence'

# Assemble ALL feature columns
ALL_FEATURES = (
    FINAL_ENV_FEATURES +           # Environmental (VIF-filtered)
    FIRE_HISTORY_FEATURES +        # Prior-block fire history (Section 6)
    TEMPORAL_DELTA_COLS +          # Temporal deltas (Section 7)
    ['ONI_index', 'IOD_index'] +  # Climate indices (Section 8)
    INTERACTION_COLS +             # Interaction terms (Section 8)
    ['fire_season']               # Seasonal indicator
)

# Remove any features that don't exist in DataFrame
ALL_FEATURES = [f for f in ALL_FEATURES if f in df_all.columns]

# Remove duplicates while preserving order
seen = set()
ALL_FEATURES = [f for f in ALL_FEATURES if f not in seen and not seen.add(f)]

# Verify NO current-block fire params are in feature set
CURRENT_FIRE_COLS = ['fire_occurrence', 'total_fire_count', 'FRP_mean_MW', 'FRP_max_MW',
                     'burned_area_binary', 'burn_month_count', 'FIRMS_brightness_K', 'FIRMS_confidence']
leaked = [f for f in ALL_FEATURES if f in CURRENT_FIRE_COLS]
assert len(leaked) == 0, f"❌ LEAKAGE DETECTED! These current-block fire cols are in features: {leaked}"

print(f"{'═'*60}")
print(f"  FINAL FEATURE SET: {len(ALL_FEATURES)} features")
print(f"{'═'*60}")
print(f"\n  🌍 Environmental (VIF-filtered): {len(FINAL_ENV_FEATURES)}")
for f in FINAL_ENV_FEATURES:
    print(f"     • {f}")
print(f"\n  🔥 Prior-Block Fire History: {len(FIRE_HISTORY_FEATURES)}")
for f in FIRE_HISTORY_FEATURES:
    print(f"     • {f}")
print(f"\n  📈 Temporal Deltas: {len(TEMPORAL_DELTA_COLS)}")
print(f"  🌊 Climate Indices: ONI_index, IOD_index")
print(f"  ✖️  Interactions: {len(INTERACTION_COLS)}")
print(f"  📅 Seasonal: fire_season")
print(f"\n  🎯 TARGET: {TARGET_COL}")
print(f"  ⚠️  Leakage check: {'✅ PASSED' if len(leaked)==0 else '❌ FAILED'}")

# Class distribution per block
print(f"\n  Class distribution:")
for bid in sorted(df_all['block_id'].unique()):
    block = df_all[df_all['block_id'] == bid]
    n1 = (block[TARGET_COL] == 1).sum()
    n0 = (block[TARGET_COL] == 0).sum()
    print(f"    Block {bid}: 🔥 {n1:,} | ❌ {n0:,} | fire_rate={n1/(n1+n0):.1%}")

════════════════════════════════════════════════════════════
  FINAL FEATURE SET: 34 features
════════════════════════════════════════════════════════════

  🌍 Environmental (VIF-filtered): 8
     • FWI_proxy
     • NDVI_min
     • aspect_deg
     • elevation
     • land_cover
     • population_density
     • precip_total_mm
     • terrain_x_veg

  🔥 Prior-Block Fire History: 10
     • prev_fire_occurrence
     • prev_total_fire_count
     • prev_FRP_mean_MW
     • prev_FRP_max_MW
     • prev_burned_area_binary
     • prev_burn_month_count
     • prev_FIRMS_brightness_K
     • fire_recurrence
     • fire_trend_delta
     • neighbor_fire_5km

  📈 Temporal Deltas: 8
  🌊 Climate Indices: ONI_index, IOD_index
  ✖️  Interactions: 5
  📅 Seasonal: fire_season

  🎯 TARGET: fire_occurrence
  ⚠️  Leakage check: ✅ PASSED

  Class distribution:
    Block 1: 🔥 2,765 | ❌ 4,617 | fire_rate=37.5%
    Block 2: 🔥 2,837 | ❌ 4,615 | fire_rate=38.1%
    Block 3: 🔥 2,866 | ❌ 4,621 | fire_rate=38.3%
    Bloc

---
## Section 10: Train/Test Split — Blocks 1–3 Train, Block 4 Test (Chronological)

> ⚠️ **Strict temporal split** — no random shuffling, no future data leakage.
> The model is trained on 2003–2020 and evaluated on 2021–2025.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Section 10: Train/Test Split (Chronological Temporal)
# ═══════════════════════════════════════════════════════════════

# Strict temporal split: Blocks 1-3 = TRAIN, Block 4 = TEST
train_mask = df_all['block_id'].isin([1, 2, 3])
test_mask  = df_all['block_id'] == 4

# Handle NaN/Inf in features
df_all[ALL_FEATURES] = df_all[ALL_FEATURES].replace([np.inf, -np.inf], np.nan)
df_all[ALL_FEATURES] = df_all[ALL_FEATURES].fillna(0)

X_train = df_all.loc[train_mask, ALL_FEATURES].values
y_train = df_all.loc[train_mask, TARGET_COL].values.astype(int)

X_test = df_all.loc[test_mask, ALL_FEATURES].values
y_test = df_all.loc[test_mask, TARGET_COL].values.astype(int)

# Store block IDs for later analysis
train_blocks = df_all.loc[train_mask, 'block_id'].values
test_blocks  = df_all.loc[test_mask, 'block_id'].values

print(f"{'═'*60}")
print(f"  TEMPORAL TRAIN/TEST SPLIT")
print(f"{'═'*60}")
print(f"  TRAIN (Blocks 1-3): X={X_train.shape}, y={y_train.shape}")
print(f"    🔥 fire={y_train.sum():,} | ❌ no_fire={(y_train==0).sum():,} | rate={y_train.mean():.1%}")
print(f"  TEST  (Block 4):    X={X_test.shape}, y={y_test.shape}")
print(f"    🔥 fire={y_test.sum():,} | ❌ no_fire={(y_test==0).sum():,} | rate={y_test.mean():.1%}")
print(f"\n  Features: {len(ALL_FEATURES)}")
print(f"  ⚠️  NO shuffling — strict chronological order")
print(f"{'═'*60}")

════════════════════════════════════════════════════════════
  TEMPORAL TRAIN/TEST SPLIT
════════════════════════════════════════════════════════════
  TRAIN (Blocks 1-3): X=(22321, 34), y=(22321,)
    🔥 fire=8,468 | ❌ no_fire=13,853 | rate=37.9%
  TEST  (Block 4):    X=(6895, 34), y=(6895,)
    🔥 fire=2,262 | ❌ no_fire=4,633 | rate=32.8%

  Features: 34
  ⚠️  NO shuffling — strict chronological order
════════════════════════════════════════════════════════════


## Section 11: SMOTE Oversampling on Training Set Only

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Section 11: SMOTE Oversampling (Training Set Only)
# ═══════════════════════════════════════════════════════════════
from imblearn.over_sampling import SMOTE

print(f"Before SMOTE: fire={y_train.sum():,}, no_fire={(y_train==0).sum():,}")

smote = SMOTE(sampling_strategy='auto', k_neighbors=5, random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print(f"After SMOTE:  fire={y_train_sm.sum():,}, no_fire={(y_train_sm==0).sum():,}")
print(f"  Train samples: {len(X_train)} → {len(X_train_sm)} ({len(X_train_sm)-len(X_train):+,})")
print(f"  ⚠️  SMOTE applied ONLY to training set — test set untouched")

Before SMOTE: fire=8,468, no_fire=13,853
After SMOTE:  fire=13,853, no_fire=13,853
  Train samples: 22321 → 27706 (+5,385)
  ⚠️  SMOTE applied ONLY to training set — test set untouched


## Section 12: StandardScaler — Fit on Train, Transform Test + Checkpoint Save

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Section 12: StandardScaler + SAVE CHECKPOINT TO DISK
# ═══════════════════════════════════════════════════════════════
from sklearn.preprocessing import StandardScaler
import joblib, json, gc

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_sm)
X_test_scaled  = scaler.transform(X_test)

# ── SAVE EVERYTHING TO DISK so kernel crashes don't lose work ──
CKPT = f'{PROJECT_DIR}/data'
np.save(f'{CKPT}/X_train_scaled.npy', X_train_scaled)
np.save(f'{CKPT}/X_test_scaled.npy', X_test_scaled)
np.save(f'{CKPT}/y_train_sm.npy', y_train_sm)
np.save(f'{CKPT}/y_test.npy', y_test)
with open(f'{CKPT}/ALL_FEATURES.json', 'w') as f:
    json.dump(ALL_FEATURES, f)
joblib.dump(scaler, f'{PROJECT_DIR}/models/scaler.joblib')

print(f"✅ StandardScaler fitted on {X_train_scaled.shape[0]:,} training samples")
print(f"   Feature means (train): min={X_train_scaled.mean(axis=0).min():.4f}, max={X_train_scaled.mean(axis=0).max():.4f}")
print(f"   Feature stds (train):  min={X_train_scaled.std(axis=0).min():.4f}, max={X_train_scaled.std(axis=0).max():.4f}")
print(f"\n💾 CHECKPOINT SAVED — preprocessed data on disk!")
print(f"   {CKPT}/X_train_scaled.npy  ({X_train_scaled.shape})")
print(f"   {CKPT}/X_test_scaled.npy   ({X_test_scaled.shape})")
print(f"   {CKPT}/y_train_sm.npy, y_test.npy")
print(f"   {CKPT}/ALL_FEATURES.json   ({len(ALL_FEATURES)} features)")
gc.collect()

✅ StandardScaler fitted on 27,706 training samples
   Feature means (train): min=-0.0000, max=0.0000
   Feature stds (train):  min=0.0000, max=1.0000

💾 CHECKPOINT SAVED — preprocessed data on disk!
   /home/yaswanth/Desktop/ors project/data/X_train_scaled.npy  ((27706, 34))
   /home/yaswanth/Desktop/ors project/data/X_test_scaled.npy   ((6895, 34))
   /home/yaswanth/Desktop/ors project/data/y_train_sm.npy, y_test.npy
   /home/yaswanth/Desktop/ors project/data/ALL_FEATURES.json   (34 features)


32

---
## 🏗️ Phase C: Model Training — 5 Base Models with Optuna Hyperparameter Tuning

> **5 diverse classifiers**, each tuned with Optuna:
> - **Tree-based**: Random Forest, XGBoost, CatBoost, LightGBM
> - **Neural**: MLP
> - All evaluated on Block 4 (2021–2025) unseen test set

## Section 13: Base Model 1 — Random Forest with Optuna

In [14]:
# ═══════════════════════════════════════════════════════════════
# 🔄 RESTORE FROM CHECKPOINT — Run this after kernel restart!
# Loads preprocessed data + any already-trained models from disk
# ═══════════════════════════════════════════════════════════════
import os, json, gc, warnings
import numpy as np
import joblib
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score, f1_score
warnings.filterwarnings('ignore')

PROJECT_DIR = '/home/yaswanth/Desktop/ors project'
CKPT = f'{PROJECT_DIR}/data'

# Load preprocessed data
X_train_scaled = np.load(f'{CKPT}/X_train_scaled.npy')
X_test_scaled  = np.load(f'{CKPT}/X_test_scaled.npy')
y_train_sm     = np.load(f'{CKPT}/y_train_sm.npy')
y_test         = np.load(f'{CKPT}/y_test.npy')
with open(f'{CKPT}/ALL_FEATURES.json') as f:
    ALL_FEATURES = json.load(f)
scaler = joblib.load(f'{PROJECT_DIR}/models/scaler.joblib')

N_OPTUNA_TRIALS = 25
CV_FOLDS = 5
RANDOM_STATE = 42

# Try loading any already-trained models
saved_models = {}
model_files = {
    'rf': 'rf_model.joblib',
    'xgb': 'xgb_model.joblib', 
    'cat': 'cat_model.joblib',
    'lgb': 'lgb_model.joblib',
    'mlp': 'mlp_model.joblib'
}
for key, fname in model_files.items():
    path = f'{PROJECT_DIR}/models/{fname}'
    if os.path.exists(path):
        saved_models[key] = joblib.load(path)

print(f"✅ CHECKPOINT RESTORED!")
print(f"   X_train: {X_train_scaled.shape}, X_test: {X_test_scaled.shape}")
print(f"   y_train: {y_train_sm.shape}, y_test: {y_test.shape}")
print(f"   Features: {len(ALL_FEATURES)}")
print(f"\n   Models on disk: {list(saved_models.keys()) if saved_models else 'NONE yet'}")

if saved_models:
    print(f"\n   📊 Existing model TEST accuracy:")
    for key, model in saved_models.items():
        pred = model.predict(X_test_scaled)
        acc = accuracy_score(y_test, pred)
        f1 = f1_score(y_test, pred)
        print(f"     {key:>5s}: Accuracy={acc*100:.2f}%, F1={f1:.4f}")
gc.collect()

✅ CHECKPOINT RESTORED!
   X_train: (27706, 34), X_test: (6895, 34)
   y_train: (27706,), y_test: (6895,)
   Features: 34

   Models on disk: NONE yet


44

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Section 13: Base Model 1 — Random Forest + Optuna
# ═══════════════════════════════════════════════════════════════
import gc
from sklearn.ensemble import RandomForestClassifier

if 'rf' in saved_models:
    rf_model = saved_models['rf']
    print("✅ RF loaded from disk — SKIPPING training")
else:
    def rf_objective(trial):
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 100, 800),
            'max_depth': trial.suggest_int('max_depth', 5, 30),
            'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
            'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
            'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2']),
            'random_state': RANDOM_STATE, 'n_jobs': 2
        }
        model = RandomForestClassifier(**params)
        cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
        return cross_val_score(model, X_train_scaled, y_train_sm, cv=cv, scoring='roc_auc', n_jobs=2).mean()

    print("⏳ Tuning Random Forest (25 trials)...")
    rf_study = optuna.create_study(direction='maximize', study_name='RF')
    rf_study.optimize(rf_objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)
    print(f"\n✅ Best RF AUC-ROC (CV): {rf_study.best_value:.4f}")

    rf_model = RandomForestClassifier(**rf_study.best_params, random_state=RANDOM_STATE, n_jobs=2)
    rf_model.fit(X_train_scaled, y_train_sm)
    
    # 💾 SAVE IMMEDIATELY
    joblib.dump(rf_model, f'{PROJECT_DIR}/models/rf_model.joblib')
    del rf_study; gc.collect()
    print(f"   💾 Saved to disk!")

# Show test accuracy
rf_acc = accuracy_score(y_test, rf_model.predict(X_test_scaled))
rf_f1 = f1_score(y_test, rf_model.predict(X_test_scaled))
print(f"   🌲 RF Test Accuracy: {rf_acc*100:.2f}% | F1: {rf_f1:.4f}")

⏳ Tuning Random Forest (25 trials)...


Best trial: 19. Best value: 0.958766: 100%|██████████| 25/25 [14:24<00:00, 34.59s/it]



✅ Best RF AUC-ROC (CV): 0.9588
   💾 Saved to disk!
   🌲 RF Test Accuracy: 93.68% | F1: 0.8974


In [16]:
# ═══════════════════════════════════════════════════════════════
# 📊 RF Individual Accuracy on Test Dataset (Block 4: 2021–2025)
# ═══════════════════════════════════════════════════════════════
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix, classification_report

rf_pred = rf_model.predict(X_test_scaled)
rf_proba = rf_model.predict_proba(X_test_scaled)[:, 1]
tn, fp, fn, tp = confusion_matrix(y_test, rf_pred).ravel()

print("═" * 60)
print("  🌲 RANDOM FOREST — TEST RESULTS (Block 4, unseen data)")
print("═" * 60)
print(f"  Accuracy  : {accuracy_score(y_test, rf_pred)*100:.2f}%")
print(f"  F1-Score  : {f1_score(y_test, rf_pred):.4f}")
print(f"  AUC-ROC   : {roc_auc_score(y_test, rf_proba):.4f}")
print(f"  Precision : {tp/(tp+fp)*100:.2f}%")
print(f"  Recall    : {tp/(tp+fn)*100:.2f}%")
print(f"  Specificity: {tn/(tn+fp)*100:.2f}%")
print(f"\n  Confusion Matrix:")
print(f"    TP={tp:,}  FP={fp:,}")
print(f"    FN={fn:,}  TN={tn:,}")
print(f"\n  Test samples: {len(y_test):,} | Fire: {int(y_test.sum()):,} | No-fire: {int(len(y_test)-y_test.sum()):,}")
print("═" * 60)

════════════════════════════════════════════════════════════
  🌲 RANDOM FOREST — TEST RESULTS (Block 4, unseen data)
════════════════════════════════════════════════════════════
  Accuracy  : 93.68%
  F1-Score  : 0.8974
  AUC-ROC   : 0.9735
  Precision : 95.97%
  Recall    : 84.26%
  Specificity: 98.27%

  Confusion Matrix:
    TP=1,906  FP=80
    FN=356  TN=4,553

  Test samples: 6,895 | Fire: 2,262 | No-fire: 4,633
════════════════════════════════════════════════════════════


## Section 14: Base Model 2 — XGBoost with Optuna

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Section 14: Base Model 2 — XGBoost + Optuna
# ═══════════════════════════════════════════════════════════════
from xgboost import XGBClassifier

if 'xgb' in saved_models:
    xgb_model = saved_models['xgb']
    print("✅ XGBoost loaded from disk — SKIPPING training")
else:
    def xgb_objective(trial):
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
            'max_depth': trial.suggest_int('max_depth', 3, 12),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
            'gamma': trial.suggest_float('gamma', 0, 5),
            'reg_alpha': trial.suggest_float('reg_alpha', 0, 10),
            'reg_lambda': trial.suggest_float('reg_lambda', 0, 10),
            'tree_method': 'hist', 'eval_metric': 'auc',
            'random_state': RANDOM_STATE, 'verbosity': 0
        }
        model = XGBClassifier(**params)
        cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
        return cross_val_score(model, X_train_scaled, y_train_sm, cv=cv, scoring='roc_auc', n_jobs=2).mean()

    print("⏳ Tuning XGBoost (25 trials)...")
    xgb_study = optuna.create_study(direction='maximize', study_name='XGB')
    xgb_study.optimize(xgb_objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)
    print(f"\n✅ Best XGB AUC-ROC (CV): {xgb_study.best_value:.4f}")

    xgb_params = {**xgb_study.best_params, 'tree_method': 'hist', 'eval_metric': 'auc',
                  'random_state': RANDOM_STATE, 'verbosity': 0}
    xgb_model = XGBClassifier(**xgb_params)
    xgb_model.fit(X_train_scaled, y_train_sm)
    
    joblib.dump(xgb_model, f'{PROJECT_DIR}/models/xgb_model.joblib')
    del xgb_study; gc.collect()
    print(f"   💾 Saved to disk!")

xgb_acc = accuracy_score(y_test, xgb_model.predict(X_test_scaled))
xgb_f1 = f1_score(y_test, xgb_model.predict(X_test_scaled))
print(f"   ⚡ XGB Test Accuracy: {xgb_acc*100:.2f}% | F1: {xgb_f1:.4f}")

⏳ Tuning XGBoost (25 trials)...


Best trial: 23. Best value: 0.960185: 100%|██████████| 25/25 [01:33<00:00,  3.72s/it]



✅ Best XGB AUC-ROC (CV): 0.9602
   💾 Saved to disk!
   ⚡ XGB Test Accuracy: 93.11% | F1: 0.8863


## Section 15: Base Model 3 — CatBoost with Optuna

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Section 15: Base Model 3 — CatBoost + Optuna (MEMORY-SAFE)
# ═══════════════════════════════════════════════════════════════
from catboost import CatBoostClassifier
import gc

if 'cat' in saved_models:
    cat_model = saved_models['cat']
    print("✅ CatBoost loaded from disk — SKIPPING training")
else:
    gc.collect()  # Free memory before heavy training
    
    def cat_objective(trial):
        params = {
            'iterations': trial.suggest_int('iterations', 200, 600),  # Reduced max
            'depth': trial.suggest_int('depth', 4, 8),                # Reduced max
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10),
            'bagging_temperature': trial.suggest_float('bagging_temperature', 0, 5),
            'random_strength': trial.suggest_float('random_strength', 0, 5),
            'verbose': 0, 'random_seed': RANDOM_STATE, 'task_type': 'CPU',
            'thread_count': 2  # Limit threads to save memory
        }
        model = CatBoostClassifier(**params)
        cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
        return cross_val_score(model, X_train_scaled, y_train_sm, cv=cv, scoring='roc_auc', n_jobs=1).mean()

    print("⏳ Tuning CatBoost (15 trials, memory-safe)...")
    cat_study = optuna.create_study(direction='maximize', study_name='CatBoost')
    cat_study.optimize(cat_objective, n_trials=15, show_progress_bar=True)
    print(f"\n✅ Best CatBoost AUC-ROC (CV): {cat_study.best_value:.4f}")

    cat_params = {**cat_study.best_params, 'verbose': 0, 'random_seed': RANDOM_STATE,
                  'task_type': 'CPU', 'thread_count': 2}
    cat_model = CatBoostClassifier(**cat_params)
    cat_model.fit(X_train_scaled, y_train_sm)
    
    joblib.dump(cat_model, f'{PROJECT_DIR}/models/cat_model.joblib')
    del cat_study; gc.collect()
    print(f"   💾 Saved to disk!")

cat_acc = accuracy_score(y_test, cat_model.predict(X_test_scaled))
cat_f1 = f1_score(y_test, cat_model.predict(X_test_scaled))
print(f"   🐱 CatBoost Test Accuracy: {cat_acc*100:.2f}% | F1: {cat_f1:.4f}")

⏳ Tuning CatBoost (15 trials, memory-safe)...


Best trial: 13. Best value: 0.96016: 100%|██████████| 15/15 [03:33<00:00, 14.24s/it] 



✅ Best CatBoost AUC-ROC (CV): 0.9602
   💾 Saved to disk!
   🐱 CatBoost Test Accuracy: 79.42% | F1: 0.5514


## Section 16: Base Model 4 — LightGBM with Optuna

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Section 16: Base Model 4 — LightGBM + Optuna (MEMORY-SAFE)
# ═══════════════════════════════════════════════════════════════
from lightgbm import LGBMClassifier

if 'lgb' in saved_models:
    lgb_model = saved_models['lgb']
    print("✅ LightGBM loaded from disk — SKIPPING training")
else:
    gc.collect()
    def lgb_objective(trial):
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 100, 800),
            'num_leaves': trial.suggest_int('num_leaves', 20, 100),
            'max_depth': trial.suggest_int('max_depth', 3, 12),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
            'min_child_samples': trial.suggest_int('min_child_samples', 5, 50),
            'reg_alpha': trial.suggest_float('reg_alpha', 0, 10),
            'reg_lambda': trial.suggest_float('reg_lambda', 0, 10),
            'random_state': RANDOM_STATE, 'verbosity': -1, 'n_jobs': 2
        }
        model = LGBMClassifier(**params)
        cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
        return cross_val_score(model, X_train_scaled, y_train_sm, cv=cv, scoring='roc_auc', n_jobs=2).mean()

    print("⏳ Tuning LightGBM (25 trials)...")
    lgb_study = optuna.create_study(direction='maximize', study_name='LightGBM')
    lgb_study.optimize(lgb_objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)
    print(f"\n✅ Best LightGBM AUC-ROC (CV): {lgb_study.best_value:.4f}")

    lgb_params = {**lgb_study.best_params, 'random_state': RANDOM_STATE, 'verbosity': -1, 'n_jobs': 2}
    lgb_model = LGBMClassifier(**lgb_params)
    lgb_model.fit(X_train_scaled, y_train_sm)
    
    joblib.dump(lgb_model, f'{PROJECT_DIR}/models/lgb_model.joblib')
    del lgb_study; gc.collect()
    print(f"   💾 Saved to disk!")

lgb_acc = accuracy_score(y_test, lgb_model.predict(X_test_scaled))
lgb_f1 = f1_score(y_test, lgb_model.predict(X_test_scaled))
print(f"   🌿 LightGBM Test Accuracy: {lgb_acc*100:.2f}% | F1: {lgb_f1:.4f}")

⏳ Tuning LightGBM (25 trials)...


Best trial: 21. Best value: 0.961886: 100%|██████████| 25/25 [02:21<00:00,  5.65s/it]



✅ Best LightGBM AUC-ROC (CV): 0.9619
   💾 Saved to disk!
   🌿 LightGBM Test Accuracy: 93.27% | F1: 0.8896


## Section 17: Base Model 5 — MLP Neural Network with Optuna

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Section 17: Base Model 5 — MLP Neural Network + Optuna (MEMORY-SAFE)
# ═══════════════════════════════════════════════════════════════
from sklearn.neural_network import MLPClassifier

if 'mlp' in saved_models:
    mlp_model = saved_models['mlp']
    print("✅ MLP loaded from disk — SKIPPING training")
else:
    gc.collect()
    def mlp_objective(trial):
        n_layers = trial.suggest_int('n_layers', 2, 3)
        layers = [trial.suggest_int(f'n_units_l{i}', 32, 128) for i in range(n_layers)]
        params = {
            'hidden_layer_sizes': tuple(layers),
            'activation': trial.suggest_categorical('activation', ['relu', 'tanh']),
            'alpha': trial.suggest_float('alpha', 1e-5, 1e-1, log=True),
            'learning_rate_init': trial.suggest_float('learning_rate_init', 1e-4, 1e-2, log=True),
            'batch_size': trial.suggest_categorical('batch_size', [64, 128, 256]),
            'early_stopping': True, 'validation_fraction': 0.15,
            'max_iter': 300, 'random_state': RANDOM_STATE
        }
        model = MLPClassifier(**params)
        cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
        return cross_val_score(model, X_train_scaled, y_train_sm, cv=cv, scoring='roc_auc', n_jobs=1).mean()

    print("⏳ Tuning MLP (20 trials)...")
    mlp_study = optuna.create_study(direction='maximize', study_name='MLP')
    mlp_study.optimize(mlp_objective, n_trials=20, show_progress_bar=True)
    print(f"\n✅ Best MLP AUC-ROC (CV): {mlp_study.best_value:.4f}")

    best_layers = [mlp_study.best_params[f'n_units_l{i}'] for i in range(mlp_study.best_params['n_layers'])]
    mlp_model = MLPClassifier(
        hidden_layer_sizes=tuple(best_layers),
        activation=mlp_study.best_params['activation'],
        alpha=mlp_study.best_params['alpha'],
        learning_rate_init=mlp_study.best_params['learning_rate_init'],
        batch_size=mlp_study.best_params['batch_size'],
        early_stopping=True, validation_fraction=0.15,
        max_iter=300, random_state=RANDOM_STATE
    )
    mlp_model.fit(X_train_scaled, y_train_sm)
    
    joblib.dump(mlp_model, f'{PROJECT_DIR}/models/mlp_model.joblib')
    del mlp_study; gc.collect()
    print(f"   💾 Saved to disk!")

mlp_acc = accuracy_score(y_test, mlp_model.predict(X_test_scaled))
mlp_f1 = f1_score(y_test, mlp_model.predict(X_test_scaled))
print(f"   🧠 MLP Test Accuracy: {mlp_acc*100:.2f}% | F1: {mlp_f1:.4f}")

⏳ Tuning MLP (20 trials)...


Best trial: 7. Best value: 0.942901: 100%|██████████| 20/20 [17:14<00:00, 51.73s/it]



✅ Best MLP AUC-ROC (CV): 0.9429
   💾 Saved to disk!
   🧠 MLP Test Accuracy: 79.96% | F1: 0.5887


---
## Section 18: SHAP Explainability — Feature & Group-Level Importance

In [1]:
# ═══════════════════════════════════════════════════════════════
# Section 18: SHAP Explainability  (LightGBM — native SHAP, ~1-2 min)
# ═══════════════════════════════════════════════════════════════
import shap, gc, time
t0 = time.time()

# LightGBM has native TreeSHAP — much faster than RF
print("Computing SHAP values using LightGBM model...")
explainer = shap.TreeExplainer(lgb_model)

# Small subsample — 500 samples is plenty for SHAP plots
shap_sample_size = min(500, len(X_test_scaled))
X_shap = X_test_scaled[:shap_sample_size]
raw_shap = explainer.shap_values(X_shap)

# Handle different SHAP output formats
if isinstance(raw_shap, list):
    # Binary classification: [class_0_shap, class_1_shap]
    shap_values = raw_shap[1]  # class-1 (fire)
    print(f"  SHAP returned list of {len(raw_shap)} arrays")
else:
    shap_values = raw_shap
    print(f"  SHAP returned single array")

# Ensure 2D
if shap_values.ndim == 1:
    shap_values = shap_values.reshape(1, -1)

np.save(f'{PROJECT_DIR}/data/shap_values.npy', shap_values)
print(f"  SHAP values shape: {shap_values.shape}  ({time.time()-t0:.1f}s)")

# ── 1. SHAP Summary Beeswarm Plot ──
fig = plt.figure(figsize=(12, 10))
shap.summary_plot(shap_values, X_shap, feature_names=ALL_FEATURES,
                  max_display=20, show=False)
plt.title('SHAP Feature Importance — Top 20 Features (LightGBM)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{PROJECT_DIR}/plots/shap_summary.png', dpi=200, bbox_inches='tight')
plt.show()
plt.close()

# ── 2. SHAP Bar Plot ──
fig = plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_shap, feature_names=ALL_FEATURES,
                  plot_type='bar', max_display=20, show=False)
plt.title('Mean |SHAP| Feature Importance (LightGBM)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{PROJECT_DIR}/plots/shap_bar.png', dpi=200, bbox_inches='tight')
plt.show()
plt.close()

# ── 3. Group-Level SHAP ──
FEATURE_GROUPS = {}
for f in ALL_FEATURES:
    if f.startswith('prev_') or f in ['fire_recurrence', 'fire_trend_delta', 'neighbor_fire_5km']:
        FEATURE_GROUPS[f] = '🔥 Fire History'
    elif f.startswith('delta_'):
        FEATURE_GROUPS[f] = '📈 Temporal Delta'
    elif any(x in f.lower() for x in ['temp', 'precip', 'wind', 'pressure', 'solar', 'soil', 'dewpoint']):
        FEATURE_GROUPS[f] = '🌤️ Weather'
    elif any(x in f.lower() for x in ['ndvi', 'evi', 'lai', 'fpar']):
        FEATURE_GROUPS[f] = '🌿 Vegetation'
    elif any(x in f.lower() for x in ['elevation', 'slope', 'aspect', 'tpi']):
        FEATURE_GROUPS[f] = '⛰️ Topography'
    elif any(x in f.lower() for x in ['oni', 'iod']):
        FEATURE_GROUPS[f] = '🌊 Climate Index'
    elif any(x in f.lower() for x in ['population', 'landcover']):
        FEATURE_GROUPS[f] = '👥 Human/Land'
    else:
        FEATURE_GROUPS[f] = '✖️ Interaction/Other'

group_shap = {}
for i, f in enumerate(ALL_FEATURES):
    group = FEATURE_GROUPS.get(f, 'Other')
    if group not in group_shap:
        group_shap[group] = 0
    group_shap[group] += np.abs(shap_values[:, i]).mean()

group_df = pd.Series(group_shap).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(10, 6))
colors_map = {
    '🔥 Fire History': '#ff6b35', '🌤️ Weather': '#448aff',
    '🌿 Vegetation': '#00e676', '⛰️ Topography': '#795548',
    '👥 Human/Land': '#ff9800', '📈 Temporal Delta': '#bb86fc',
    '🌊 Climate Index': '#00bcd4', '✖️ Interaction/Other': '#9e9e9e'
}
bar_colors = [colors_map.get(g, '#9e9e9e') for g in group_df.index]
group_df.plot(kind='barh', ax=ax, color=bar_colors, edgecolor='white', linewidth=0.5)
ax.set_xlabel('Mean |SHAP| Value', fontsize=12)
ax.set_title('Group-Level Feature Importance (SHAP)', fontsize=14, fontweight='bold')
ax.grid(alpha=0.2, axis='x')
plt.tight_layout()
plt.savefig(f'{PROJECT_DIR}/plots/shap_groups.png', dpi=200, bbox_inches='tight')
plt.show()
plt.close()

gc.collect()
elapsed = time.time() - t0
print(f"\n✅ SHAP analysis complete in {elapsed:.1f}s — fire history group rank: ",
      f"{'🏆 HIGHEST' if group_shap.get('🔥 Fire History', 0) == max(group_shap.values()) else 'check plots'}")

/home/yaswanth/Desktop/ors project/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Computing SHAP values using LightGBM model...


NameError: name 'lgb_model' is not defined

## Section 19: Block-wise Pattern Visualization — How Fire Drivers Evolve

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Section 19: Block-wise Pattern Visualization
# ═══════════════════════════════════════════════════════════════

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# ── Panel 1: Fire rates & FRP evolution across blocks ──
ax = axes[0, 0]
block_stats = df_all.groupby('block_id').agg({
    'fire_occurrence': 'mean',
    'FRP_mean_MW': 'mean',
    'total_fire_count': 'mean'
}).reset_index()

ax2 = ax.twinx()
ax.bar(block_stats['block_id'] - 0.15, block_stats['fire_occurrence'] * 100,
       width=0.3, color='#ff6b35', alpha=0.8, label='Fire Rate (%)')
ax2.plot(block_stats['block_id'], block_stats['FRP_mean_MW'],
         'o-', color='#ffd700', lw=2, ms=8, label='Mean FRP (MW)')
ax.set_xlabel('Block', fontsize=12)
ax.set_ylabel('Fire Rate (%)', fontsize=12, color='#ff6b35')
ax2.set_ylabel('Mean FRP (MW)', fontsize=12, color='#ffd700')
ax.set_title('Fire Metrics Evolution (2003–2025)', fontsize=13, fontweight='bold')
ax.set_xticks([1, 2, 3, 4])
ax.set_xticklabels(['B1\n2003-08', 'B2\n2009-14', 'B3\n2015-20', 'B4\n2021-25'])
ax.legend(loc='upper left', fontsize=9)
ax2.legend(loc='upper right', fontsize=9)

# ── Panel 2: Feature importance per block (RF) ──
ax = axes[0, 1]
block_importance = {}
for bid in [1, 2, 3]:
    mask = df_all['block_id'] == bid
    X_b = df_all.loc[mask, ALL_FEATURES].replace([np.inf, -np.inf], np.nan).fillna(0).values
    y_b = df_all.loc[mask, TARGET_COL].values.astype(int)
    rf_temp = RandomForestClassifier(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1)
    rf_temp.fit(scaler.transform(X_b), y_b)
    block_importance[f'Block {bid}'] = rf_temp.feature_importances_

imp_df = pd.DataFrame(block_importance, index=ALL_FEATURES)
top10 = imp_df.mean(axis=1).nlargest(10).index
imp_df.loc[top10].plot(kind='barh', ax=ax, width=0.8)
ax.set_xlabel('Feature Importance', fontsize=12)
ax.set_title('Top 10 Features Across Training Blocks', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(alpha=0.2, axis='x')

# ── Panel 3: Fire history vs weather contribution ──
ax = axes[1, 0]
fire_hist_feats = [f for f in ALL_FEATURES if f in FIRE_HISTORY_FEATURES]
weather_feats = [f for f in ALL_FEATURES if FEATURE_GROUPS.get(f) == '🌤️ Weather']
veg_feats = [f for f in ALL_FEATURES if FEATURE_GROUPS.get(f) == '🌿 Vegetation']

# Use RF feature importance
rf_imp = pd.Series(rf_model.feature_importances_, index=ALL_FEATURES)
group_imp = {
    '🔥 Fire History': rf_imp[fire_hist_feats].sum() if fire_hist_feats else 0,
    '🌤️ Weather': rf_imp[weather_feats].sum() if weather_feats else 0,
    '🌿 Vegetation': rf_imp[veg_feats].sum() if veg_feats else 0,
    '📈 Other': rf_imp.sum() - rf_imp[fire_hist_feats + weather_feats + veg_feats].sum()
}

colors_pie = ['#ff6b35', '#448aff', '#00e676', '#9e9e9e']
ax.pie(group_imp.values(), labels=group_imp.keys(), colors=colors_pie,
       autopct='%1.1f%%', startangle=90, textprops={'fontsize': 11, 'color': 'white'})
ax.set_title('Feature Group Contribution (RF)', fontsize=13, fontweight='bold')

# ── Panel 4: Predicted vs actual fire rate per block ──
ax = axes[1, 1]
for bid in sorted(df_all['block_id'].unique()):
    mask = df_all['block_id'] == bid
    X_b = df_all.loc[mask, ALL_FEATURES].replace([np.inf, -np.inf], np.nan).fillna(0).values
    y_b = df_all.loc[mask, TARGET_COL].values

    if bid <= 3:
        # Use training model predictions
        proba = rf_model.predict_proba(scaler.transform(X_b))[:, 1]
    else:
        proba = y_pred_proba

    ax.scatter(bid, y_b.mean() * 100, s=200, color='#ff6b35', zorder=5,
               label='Actual' if bid == 1 else '', edgecolors='white', linewidth=2)
    ax.scatter(bid, proba.mean() * 100, s=200, color='#448aff', zorder=5,
               marker='D', label='Predicted' if bid == 1 else '', edgecolors='white', linewidth=2)

ax.set_xlabel('Block', fontsize=12)
ax.set_ylabel('Fire Rate (%)', fontsize=12)
ax.set_title('Actual vs Predicted Fire Rate per Block', fontsize=13, fontweight='bold')
ax.set_xticks([1, 2, 3, 4])
ax.set_xticklabels(['B1\n2003-08', 'B2\n2009-14', 'B3\n2015-20', 'B4\n2021-25'])
ax.legend(fontsize=10)
ax.grid(alpha=0.2)

plt.suptitle('AGFE-Fire v4 — Block-wise Fire Pattern Analysis', fontsize=16,
             fontweight='bold', color='#ffd700', y=1.01)
plt.tight_layout()
plt.savefig(f'{PROJECT_DIR}/plots/blockwise_patterns.png', dpi=200, bbox_inches='tight',
            facecolor='black')
plt.show()
print(f"💾 Saved: {PROJECT_DIR}/plots/blockwise_patterns.png")

💾 Saved: /home/yaswanth/Desktop/ors project/plots/blockwise_patterns.png


## Section 20: Save All Models & Artifacts Locally

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Section 20: Save All Artifacts Locally
# ═══════════════════════════════════════════════════════════════
import os, joblib

saved_files = []

# Models
model_files = [
    ('scaler.joblib', scaler),
    ('rf_model.joblib', rf_model),
    ('xgboost_model.joblib', xgb_model),
    ('catboost_model.joblib', cat_model),
    ('lightgbm_model.joblib', lgb_model),
    ('mlp_model.joblib', mlp_model),
]
for fname, obj in model_files:
    path = f'{PROJECT_DIR}/models/{fname}'
    joblib.dump(obj, path)
    size = os.path.getsize(path)
    saved_files.append((path, size))

# Data
data_files = {
    'WG_AGFE_all_blocks_engineered.csv': df_all,
}
if 'shap_values' in dir():
    data_files['shap_values.npy'] = shap_values

for fname, data in data_files.items():
    path = f'{PROJECT_DIR}/data/{fname}'
    if fname.endswith('.csv'):
        data.to_csv(path, index=False)
    else:
        np.save(path, data)
    size = os.path.getsize(path)
    saved_files.append((path, size))

# Feature list
feat_path = f'{PROJECT_DIR}/data/feature_list.txt'
with open(feat_path, 'w') as f:
    for feat in ALL_FEATURES:
        group = FEATURE_GROUPS.get(feat, 'Other') if 'FEATURE_GROUPS' in dir() else 'N/A'
        f.write(f"{feat}\t{group}\n")
saved_files.append((feat_path, os.path.getsize(feat_path)))

# Print summary
print("═" * 70)
print("  ALL ARTIFACTS SAVED LOCALLY")
print("═" * 70)
total_size = 0
for path, size in saved_files:
    size_str = f"{size/1024:.1f} KB" if size < 1024*1024 else f"{size/1024/1024:.1f} MB"
    print(f"  💾 {path.replace(PROJECT_DIR+'/', ''):<45s} {size_str:>10s}")
    total_size += size
print(f"\n  📦 Total: {total_size/1024/1024:.1f} MB across {len(saved_files)} files")
print(f"  📂 Location: {PROJECT_DIR}/")
print("═" * 70)

══════════════════════════════════════════════════════════════════════
  ALL ARTIFACTS SAVED LOCALLY
══════════════════════════════════════════════════════════════════════
  💾 models/scaler.joblib                              1.4 KB
  💾 models/rf_model.joblib                           99.4 MB
  💾 models/xgboost_model.joblib                       2.6 MB
  💾 models/catboost_model.joblib                   1003.5 KB
  💾 models/lightgbm_model.joblib                      4.5 MB
  💾 models/mlp_model.joblib                         165.0 KB
  💾 models/meta_learner.joblib                        7.8 KB
  💾 data/WG_AGFE_all_blocks_engineered.csv           22.3 MB
  💾 data/oof_train.npy                                1.1 MB
  💾 data/oof_test.npy                               269.5 KB
  💾 data/shap_values.npy                            132.9 KB
  💾 data/feature_list.txt                             1.2 KB

  📦 Total: 131.5 MB across 12 files
  📂 Location: /home/yaswanth/Desktop/ors project/
═════════

---
## 📊 Section 21: Presentation — Model Comparison & Data Analysis Plots

> **8 publication-quality figures** for comparing all 5 models on Block 4 test set.
> Each figure saved at 200 DPI in the `plots/` folder.

In [32]:
# ═══════════════════════════════════════════════════════════════
# Section 21a: Compute All Metrics for 5 Models (No Ensemble)
# ═══════════════════════════════════════════════════════════════
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, roc_auc_score, average_precision_score,
    f1_score, matthews_corrcoef, cohen_kappa_score,
    brier_score_loss, confusion_matrix, classification_report,
    roc_curve, precision_recall_curve, recall_score, precision_score
)

MODELS = {
    'Random Forest': rf_model,
    'XGBoost': xgb_model,
    'CatBoost': cat_model,
    'LightGBM': lgb_model,
    'MLP': mlp_model
}

MODEL_COLORS = {
    'Random Forest': '#4caf50',
    'XGBoost': '#448aff',
    'CatBoost': '#e91e63',
    'LightGBM': '#bb86fc',
    'MLP': '#ff9800'
}

# Compute predictions and full metrics for each model
model_preds = {}
model_metrics = {}

for name, model in MODELS.items():
    pred = model.predict(X_test_scaled)
    proba = model.predict_proba(X_test_scaled)[:, 1]
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0

    model_preds[name] = {'pred': pred, 'proba': proba}
    model_metrics[name] = {
        'Accuracy': accuracy_score(y_test, pred),
        'AUC-ROC': roc_auc_score(y_test, proba),
        'AUC-PR': average_precision_score(y_test, proba),
        'F1-Score': f1_score(y_test, pred),
        'Precision': precision_score(y_test, pred),
        'Recall (Sensitivity)': sens,
        'Specificity': spec,
        'MCC': matthews_corrcoef(y_test, pred),
        "Cohen's κ": cohen_kappa_score(y_test, pred),
        'TSS': sens + spec - 1,
        'Brier Score': brier_score_loss(y_test, proba),
    }

metrics_df = pd.DataFrame(model_metrics).T.round(4)

# Save metrics
metrics_df.to_csv(f'{PROJECT_DIR}/results/model_comparison_metrics.csv')

print("═" * 85)
print("  📊 FULL MODEL COMPARISON — Block 4 Test Set (2021–2025)")
print("═" * 85)
print(metrics_df.to_string())
print("\n  🏆 Best per metric:")
for col in metrics_df.columns:
    best = metrics_df[col].idxmin() if col == 'Brier Score' else metrics_df[col].idxmax()
    print(f"    {col:22s}: {best} ({metrics_df.loc[best, col]:.4f})")
print(f"\n💾 Saved: {PROJECT_DIR}/results/model_comparison_metrics.csv")

═════════════════════════════════════════════════════════════════════════════════════
  📊 FULL MODEL COMPARISON — Block 4 Test Set (2021–2025)
═════════════════════════════════════════════════════════════════════════════════════
               Accuracy  AUC-ROC  AUC-PR  F1-Score  Precision  Recall (Sensitivity)  Specificity     MCC  Cohen's κ     TSS  Brier Score
Random Forest    0.9368   0.9735  0.9602    0.8974     0.9597                0.8426       0.9827  0.8557     0.8519  0.8253       0.0680
XGBoost          0.9311   0.9759  0.9616    0.8863     0.9661                0.8187       0.9860  0.8433     0.8374  0.8047       0.0540
CatBoost         0.7942   0.9375  0.8980    0.5514     0.9678                0.3855       0.9937  0.5283     0.4483  0.3792       0.1417
LightGBM         0.9327   0.9761  0.9627    0.8896     0.9629                0.8267       0.9845  0.8467     0.8416  0.8112       0.0522
MLP              0.7996   0.8841  0.8155    0.5887     0.9007                0.4372   

In [33]:
# ═══════════════════════════════════════════════════════════════
# Section 21b: Accuracy & F1 Grouped Bar Chart
# ═══════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(12, 6))
names = list(MODELS.keys())
x = np.arange(len(names))
w = 0.35
acc_vals = [model_metrics[n]['Accuracy'] * 100 for n in names]
f1_vals  = [model_metrics[n]['F1-Score'] * 100 for n in names]

bars1 = ax.bar(x - w/2, acc_vals, w, label='Accuracy (%)', color='#448aff', edgecolor='white', linewidth=0.8, zorder=3)
bars2 = ax.bar(x + w/2, f1_vals, w, label='F1-Score (%)', color='#e91e63', edgecolor='white', linewidth=0.8, zorder=3)

for bars in [bars1, bars2]:
    for b in bars:
        ax.annotate(f'{b.get_height():.1f}%', xy=(b.get_x() + b.get_width()/2, b.get_height()),
                     xytext=(0, 5), textcoords='offset points', ha='center', va='bottom',
                     fontsize=10, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(names, fontsize=12)
ax.set_ylabel('Score (%)', fontsize=13)
ax.set_title('Model Accuracy & F1-Score Comparison — AGFE-Fire v4', fontsize=15, fontweight='bold')
ax.set_ylim(0, 110)
ax.legend(fontsize=12, loc='upper right')
ax.grid(axis='y', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
fig.savefig(f'{PROJECT_DIR}/plots/accuracy_f1_comparison.png', dpi=200, bbox_inches='tight')
print("✅ Saved: plots/accuracy_f1_comparison.png")

✅ Saved: plots/accuracy_f1_comparison.png


In [34]:
# ═══════════════════════════════════════════════════════════════
# Section 21c: ROC Curves — All 5 Models
# ═══════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(9, 8))
for name in MODELS:
    fpr, tpr, _ = roc_curve(y_test, model_preds[name]['proba'])
    auc_val = model_metrics[name]['AUC-ROC']
    ax.plot(fpr, tpr, label=f"{name} (AUC = {auc_val:.4f})",
            color=MODEL_COLORS[name], linewidth=2.2)

ax.plot([0, 1], [0, 1], 'k--', alpha=0.4, linewidth=1)
ax.set_xlabel('False Positive Rate', fontsize=13)
ax.set_ylabel('True Positive Rate', fontsize=13)
ax.set_title('ROC Curves — 5-Model Comparison', fontsize=15, fontweight='bold')
ax.legend(fontsize=11, loc='lower right', framealpha=0.9)
ax.grid(alpha=0.25)
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)
plt.tight_layout()
fig.savefig(f'{PROJECT_DIR}/plots/roc_curves_all_models.png', dpi=200, bbox_inches='tight')
print("✅ Saved: plots/roc_curves_all_models.png")

✅ Saved: plots/roc_curves_all_models.png


In [35]:
# ═══════════════════════════════════════════════════════════════
# Section 21d: Confusion Matrices — 5-Model Grid
# ═══════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 5, figsize=(26, 5))
for i, (name, color) in enumerate(MODEL_COLORS.items()):
    cm = confusion_matrix(y_test, model_preds[name]['pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i],
                xticklabels=['No Fire', 'Fire'], yticklabels=['No Fire', 'Fire'],
                annot_kws={'size': 14}, linewidths=0.5)
    axes[i].set_title(name, fontsize=13, fontweight='bold')
    axes[i].set_ylabel('True' if i == 0 else '', fontsize=12)
    axes[i].set_xlabel('Predicted', fontsize=12)

fig.suptitle('Confusion Matrices — All Models on Block 4 Test Set', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
fig.savefig(f'{PROJECT_DIR}/plots/confusion_matrices_grid.png', dpi=200, bbox_inches='tight')
print("✅ Saved: plots/confusion_matrices_grid.png")

✅ Saved: plots/confusion_matrices_grid.png


In [36]:
# ═══════════════════════════════════════════════════════════════
# Section 21e: Precision–Recall Curves — All 5 Models
# ═══════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(9, 8))
for name in MODELS:
    prec_vals, rec_vals, _ = precision_recall_curve(y_test, model_preds[name]['proba'])
    ap = model_metrics[name]['AUC-PR']
    ax.plot(rec_vals, prec_vals, label=f"{name} (AP = {ap:.4f})",
            color=MODEL_COLORS[name], linewidth=2.2)

fire_prevalence = y_test.mean()
ax.axhline(fire_prevalence, color='gray', linestyle='--', alpha=0.5,
           label=f'Baseline (prevalence = {fire_prevalence:.3f})')
ax.set_xlabel('Recall', fontsize=13)
ax.set_ylabel('Precision', fontsize=13)
ax.set_title('Precision–Recall Curves — 5-Model Comparison', fontsize=15, fontweight='bold')
ax.legend(fontsize=11, loc='upper right', framealpha=0.9)
ax.grid(alpha=0.25)
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.05)
plt.tight_layout()
fig.savefig(f'{PROJECT_DIR}/plots/precision_recall_curves.png', dpi=200, bbox_inches='tight')
print("✅ Saved: plots/precision_recall_curves.png")

✅ Saved: plots/precision_recall_curves.png


In [37]:
# ═══════════════════════════════════════════════════════════════
# Section 21f: Radar / Spider Chart — Multi-Metric Comparison
# ═══════════════════════════════════════════════════════════════
radar_metrics = ['Accuracy', 'AUC-ROC', 'F1-Score', 'Precision', 'Recall (Sensitivity)', 'Specificity', 'MCC', "Cohen's κ"]
N = len(radar_metrics)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]   # close the polygon

fig, ax = plt.subplots(figsize=(9, 9), subplot_kw=dict(polar=True))
for name in MODELS:
    vals = [model_metrics[name][m] for m in radar_metrics]
    vals += vals[:1]
    ax.plot(angles, vals, 'o-', linewidth=2, label=name, color=MODEL_COLORS[name], markersize=5)
    ax.fill(angles, vals, alpha=0.08, color=MODEL_COLORS[name])

ax.set_thetagrids(np.degrees(angles[:-1]), radar_metrics, fontsize=10)
ax.set_ylim(0, 1.05)
ax.set_title('Multi-Metric Radar — AGFE-Fire v4', fontsize=15, fontweight='bold', y=1.1)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), fontsize=10, framealpha=0.9)
ax.grid(alpha=0.3)
plt.tight_layout()
fig.savefig(f'{PROJECT_DIR}/plots/radar_chart_metrics.png', dpi=200, bbox_inches='tight')
print("✅ Saved: plots/radar_chart_metrics.png")

✅ Saved: plots/radar_chart_metrics.png


In [38]:
# ═══════════════════════════════════════════════════════════════
# Section 21g: Metric Heatmap — Full Table Visualised
# ═══════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(14, 5))
hm_df = metrics_df.drop(columns=['Brier Score'])  # lower-is-better confuses color
sns.heatmap(hm_df.astype(float), annot=True, fmt='.4f', cmap='YlGnBu',
            linewidths=0.6, ax=ax, vmin=0, vmax=1, annot_kws={'size': 11})
ax.set_title('Performance Heatmap — All Models × 10 Metrics', fontsize=15, fontweight='bold')
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=12)
ax.set_xticklabels(ax.get_xticklabels(), rotation=25, ha='right', fontsize=11)
plt.tight_layout()
fig.savefig(f'{PROJECT_DIR}/plots/metric_heatmap.png', dpi=200, bbox_inches='tight')
print("✅ Saved: plots/metric_heatmap.png")

✅ Saved: plots/metric_heatmap.png


In [39]:
# ═══════════════════════════════════════════════════════════════
# Section 21h: Feature Importance — Top 15 Across Tree Models
# ═══════════════════════════════════════════════════════════════
tree_models = {
    'Random Forest': rf_model,
    'XGBoost': xgb_model,
    'LightGBM': lgb_model,
}

fig, axes = plt.subplots(1, 3, figsize=(22, 7))
for i, (name, model) in enumerate(tree_models.items()):
    imps = model.feature_importances_
    idx = np.argsort(imps)[-15:]
    axes[i].barh(np.array(ALL_FEATURES)[idx], imps[idx],
                 color=MODEL_COLORS[name], edgecolor='white', linewidth=0.5)
    axes[i].set_title(f'{name} — Top 15 Features', fontsize=13, fontweight='bold')
    axes[i].set_xlabel('Importance', fontsize=11)
    axes[i].tick_params(axis='y', labelsize=10)
    axes[i].invert_yaxis()

fig.suptitle('Feature Importance Comparison — Tree-Based Models', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
fig.savefig(f'{PROJECT_DIR}/plots/feature_importance_comparison.png', dpi=200, bbox_inches='tight')
print("✅ Saved: plots/feature_importance_comparison.png")

✅ Saved: plots/feature_importance_comparison.png


---
## 📋 Summary & Future Goals

### ✅ Current Results (5 Individual Base Models)
| Model | Accuracy | F1-Score | AUC-ROC |
|-------|----------|----------|---------|
| Random Forest | 93.68 % | 0.8981 | 0.9739 |
| XGBoost | 93.23 % | 0.8888 | 0.9765 |
| LightGBM | 93.52 % | 0.8941 | 0.9765 |
| CatBoost | 79.42 % | 0.4892 | 0.9586 |
| MLP | 79.96 % | 0.4403 | 0.8082 |

### 🔮 Future Goals
1. **Ensemble / Stacking Methods** — OOF-based meta-learner (Ridge, Logistic) to combine the 5 base models for potentially higher accuracy.
2. **Future Predictions** — Apply the trained models to new unseen data (e.g., 2025–2026 satellite imagery) for real-time fire vulnerability mapping.
3. **Try Different Datasets** — Extend the AGFE-Fire framework to other regions beyond the Western Ghats, or incorporate additional data sources (Sentinel-2, climate reanalysis, socio-economic layers).
4. **Threshold Optimisation** — Improve CatBoost & MLP by tuning classification thresholds (Youden's J, cost-sensitive).
5. **Deep Learning Extensions** — 1-D CNN / LSTM on the temporal block sequence for sequential fire risk modelling.